# 🎬 VideoUtil Bot — Google Colab Runner

# Don't Remove Credit Tg - @NexonBots
# Subscribe YouTube Channel For Amazing Bot https://youtube.com/@NexonBots
# Ask Doubt on telegram @NexonContactBot

Run the cells **in order**. This is a demo/testing host:

- The bot keeps exactly **one** instance — stop other copies before running.
- Paste a GitHub PAT with `repo` scope when cloning (make the repo public to skip).
- Use the keep-alive cell so Colab does not disconnect the runtime.
- Interrupt the kernel to stop the bot.

In [ ]:
# Cell 1 — system packages (FFmpeg, MediaInfo, Megatools, Git)
!apt-get -qq update > /dev/null
!apt-get -qq install -y ffmpeg mediainfo git curl ca-certificates megatools > /dev/null
!ffmpeg -version | head -n 1
!mediainfo --Version | head -n 1
!git --version

In [ ]:
# Cell 2 — clone repository (private repos ask for a PAT) and install Python deps
import getpass
import os
import subprocess

REPO_DIR = "/content/VideoUtil"
REPO_URL = "https://github.com/PrimeSujoy/VideoUtil.git"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    token = getpass.getpass("GitHub PAT (repo scope, blank if public): ").strip()
    url = f"https://{token}@github.com/PrimeSujoy/VideoUtil.git" if token else REPO_URL
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

os.chdir(REPO_DIR)
!pip install -q -r requirements.txt
print("Installed into", os.getcwd())

In [ ]:
# Cell 3 — secrets: Colab "Secrets" storage first, prompt as fallback
from getpass import getpass
import os

try:
    from google.colab import userdata
except Exception:
    userdata = None

REQUIRED = [
    "APP_ID", "API_HASH", "TG_BOT_TOKEN", "BOT_USERNAME",
    "AUTH_USERS", "LOG_CHANNEL", "DB_URL", "DB_NAME",
]
OPTIONAL = [
    "SESSION_NAME", "UPDATES_CHANNEL", "PICS", "USER_SESSION_STRING",
    "STRING_API_ID", "STRING_API_HASH", "SESSION_USER_ID",
    "UPSTREAM_REPO", "GIT_TOKEN",
]

def pick(key, required):
    value = ""
    if userdata is not None:
        try:
            value = userdata.get(key)
        except Exception:
            value = ""
    if not value and required:
        value = getpass(f"{key}: ").strip()
    return value or ""

lines = []
for key in REQUIRED:
    value = pick(key, True)
    if value:
        os.environ[key] = value
        lines.append(f"{key}={value}")
for key in OPTIONAL:
    value = pick(key, False)
    if value:
        os.environ[key] = value
        lines.append(f"{key}={value}")

with open(".env", "w", encoding="utf-8") as handle:
    handle.write("\n".join(lines) + "\n")
print(f"Wrote {len(lines)} values to .env")

In [ ]:
# Cell 4 — start the bot (Kernel → Interrupt to stop)
import subprocess
import sys

subprocess.run([sys.executable, "-m", "bot"], check=False)

In [ ]:
# Cell 5 — keep-alive: clicks the Colab connect button so the runtime stays alive
from IPython.display import display, Javascript

display(Javascript('''
function ClickConnect(){
  const btn = document.querySelector("colab-connect-button") ||
             document.querySelector("#top-toolbar > colab-connect-button");
  if (btn && btn.shadowRoot) {
    const inner = btn.shadowRoot.querySelector("button");
    if (inner) inner.click();
  }
}
setInterval(ClickConnect, 60000);
'''))
print("Keep-alive installed (clicks every 60s).")

### Notes

- **Stop:** Kernel → Interrupt, or run `Kernel → Restart runtime`.
- **Logs:** printed live in Cell 4; the bot also writes `Log.txt`.
- **Persistence:** Colab disks are ephemeral — sessions/thumbnails under
  `WORK_DIR` vanish on restart. MongoDB data is safe.
- **Always-on:** use Docker, Streamlit, Heroku, Northflank, Koyeb,
  Hugging Face or Google Cloud — see [DEPLOYMENT.md](DEPLOYMENT.md).